# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: 
Date: 

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/aaditeya/Desktop/bootcamp_aaditeya_tripathi/homework/homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

v_api = validate(df_api, ['date','close']); v_api

[*********************100%***********************]  1 of 1 completed


{'missing': [], 'shape': (64, 2), 'na_total': 0}

In [6]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-yfinance_symbol-AAPL_20260826-164410.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [7]:
SCRAPE_URL = 'https://the-internet.herokuapp.com/tables'  # TODO: replace with permitted page
headers = {'User-Agent':'AFE-Homework/1.0'}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')
v_scrape = validate(df_scrape, list(df_scrape.columns)); v_scrape

{'missing': [], 'shape': (9, 6), 'na_total': 0}

In [8]:
_ = save_csv(df_scrape, prefix='scrape', site='example', table='markets')

Saved data/raw/scrape_site-example_table-markets_20260826-164411.csv


## Documentation

### 1. Data Sources & Ingestion Parameters
* **API Ingestion (`yfinance`):**
  * **Source / Library:** Yahoo Finance via `yfinance`.
  * **Ticker / Symbol:** `AAPL`.
  * **Parameters:** `period='3mo'`, `interval='1d'`, `auto_adjust=False`.
  * **Schema / Columns Parsed:** `date`, `close`.
* **Web Scraping (`BeautifulSoup`):**
  * **Source URL:** `https://the-internet.herokuapp.com/tables`.
  * **Parser Logic:** `requests.get()` with custom User-Agent headers, parsed using `BeautifulSoup(resp.text, 'html.parser')`.
  * **Target Elements:** Extracted `<th>` header tags for column names and `<tr>`/`<td>` nodes for tabular observation rows.

### 2. Validation Rules & Logic
* **Required Column Checks:** Evaluated presence of mandatory identifiers (`date`, `close` for API; parsed table headers for web scrape) via the `validate()` helper function.
* **Missing Value (NA) Auditing:** Verified that missing value counts (`na_total == 0`) across both DataFrames.
* **Type Casting:** Cast date fields to datetime objects and price/numerical metrics to float64 to ensure correct downstream mathematical operations.

### 3. Assumptions & Operational Risks
* **API Rate Limits:** Sourcing market quotes via free endpoints is subject to daily call limits.
* **HTML Selector Fragility:** Scraping depends website redesigns, dynamic JavaScript rendering, or renamed class/tag structures will break static parsers.
* **Schema Drift:** Upstream column name modifications or missing timestamps can corrupt the automated persistence schema.

### 4. Secrets & Environment Security
* **Environment Confirmation:** `.env` is omitted from version control and verified inside `.gitignore`.
* **Template Tracking:** Only `.env.example` containing non-sensitive placeholder variables is tracked and committed to Git.